## What to Vary

In [1]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

In [2]:
from topicnet.cooking_machine import Dataset

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

from umap import UMAP
from hdbscan import HDBSCAN

from hdbscan.flat import HDBSCAN_flat

from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

import pandas as pd

In [3]:
import nltk
from nltk.corpus import stopwords
 
nltk.download('stopwords')

print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/alekseev_v/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/postnauka_noow.csv',
)

dataset.get_possible_modalities()

set()

In [7]:
dataset.get_possible_modalities()

set()

In [8]:
MAIN_MODALITY = '@word'

In [9]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,|@word материал отрицательный показатель прело...
7770.txt,7770.txt,культурный код экономика экономист александр а...,|@word культурный код экономика экономист алек...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,|@word faq наука третий класс факт эксперимент...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,|@word обрушение волна поверхность жидкость ма...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,|@word существовать ли суперсимметрия мир элем...


In [10]:
dataset._data.shape

(3446, 3)

In [11]:
dataset._data.dropna(axis=0, inplace=True)

In [12]:
dataset._data.shape

(3446, 3)

In [13]:
dataset._data['raw_text']  # TODO: say about data preproc for BERTopic (raw or preprocessed text)

id
29998.txt    материал отрицательный показатель преломление ...
7770.txt     культурный код экономика экономист александр а...
32230.txt    faq наука третий класс факт эксперимент резуль...
27293.txt    обрушение волна поверхность жидкость математик...
481.txt      существовать ли суперсимметрия мир элементарны...
                                   ...                        
49461.txt    пептидный белковый нейротоксин химик виктор це...
15983.txt    радиотелескоп земля космос астрофизик анатолий...
5069.txt     вояджер история полт два исследовательский зон...
31220.txt    феномен чайлдфри общество социолог ольга исупо...
9795.txt     шаг теория принятие решение книга необходимый ...
Name: raw_text, Length: 3446, dtype: object

In [14]:
docs = list(dataset._data['raw_text'].values)

In [15]:
docs[:3]

['материал отрицательный показатель преломление физик виктор веселаго распространение свет вещество фазовый групповой скорость метаматериалы различаться фазовый групповой скорость каков физика распространение свет вещество находить применение материал отрицательный показатель преломление рассказывать доктор физикоматематический наука виктор веселаго скорость распространяться энергия вещество обычно говорить излучение распространяться вещество со скорость n раз маленький n коэффициент преломление вещество коэффициент преломление n отношение скорость свет скорость распространение излучение вещество обычно уточняться распространяться распространение энергия распространение импульс происходить различный закон энергия распространяться со скорость называться групповой скорость много скорость свет эйнштейн сформулировать самый больший скорость излучение скорость свет кмс импульс распространяться фазовый скорость сколь угодно много скорость свет скорость входить соотношение emc фазовый группов

In [16]:
NUM_TOP_WORDS = 20

In [17]:
import torch
import transformers
import os

import json
import numpy as np

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [18]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    # ptw = np.array(mtw[1:, :])
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    # topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    return phi


def get_top_words(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T

    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in zip([-1] + list(range(NUM_TOPICS)), topic_names)
    }

    return topic_top_words


def get_dataset(topic_model, dataset, docs):
    cleaned_docs = topic_model._preprocess_text(docs)
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()
    doc_tokens = [tokenizer(doc) for doc in cleaned_docs]
    doc_texts = [
        d + f' |{MAIN_MODALITY} ' + ' '.join(t)
        for d, t in zip(dataset._data.index, doc_tokens)
    ]
    data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]

    new_dataset = pd.DataFrame(
        columns=['id', 'vw_text'],
        data=data,
    )

    return new_dataset

In [19]:
NUM_TOPICS = 50
NUM_TOP_WORDS = 20
NUM_TRAINS = 20
STOP_WORDS = stopwords.words('russian')
LANGUAGE = 'multilingual'

In [20]:
! ls ../results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [21]:
SAVE_FOLDER = os.path.join('/data_mil/shared/CompressaAI/BERTopic', 'results50', 'postnauka')

In [22]:
! mkdir -p $SAVE_FOLDER

In [23]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results50/postnauka'

In [24]:
for seed in range(NUM_TRAINS):
    print(seed)

    seed_save_folder = os.path.join(SAVE_FOLDER, str(seed))

    os.makedirs(seed_save_folder)

    keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
    mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)
    
    representation_model = {
        "KeyBERT": keybert,
        "MMR": mmr,
    }

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,

        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )
        
    topics, probs = topic_model.fit_transform(docs)
    orig_num_topics = len(set(topic_model.topics_))
    doc_embeddings = topic_model.umap_model.embedding_

    hdbscan_model = HDBSCAN_flat(doc_embeddings, n_clusters=NUM_TOPICS)
    
    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,
        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )

    topics, probs = topic_model.fit_transform(docs)
    
    new_num_topics = len(set(topic_model.topics_))
    
    assert new_num_topics < orig_num_topics
    assert new_num_topics == NUM_TOPICS + 1
    assert topic_model.c_tf_idf_.shape[0] == new_num_topics
    
    phi = get_phi(topic_model)
    top_words = get_top_words(topic_model)
    new_dataset = get_dataset(topic_model, dataset, docs)
    
    phi.to_csv(f'{seed_save_folder}/phi.csv')
    
    with open(f'{seed_save_folder}/top_words.json', 'w') as f:
        f.write(
            json.dumps(
                top_words, indent=4, ensure_ascii=False
            )
        )
    
    new_dataset.to_csv(f'{seed_save_folder}/dataset.csv')

    del topic_model, phi, new_dataset

2024-03-30 02:50:05,498 - BERTopic - Embedding - Transforming documents to embeddings.


0


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2024-03-30 02:50:17,084 - BERTopic - Embedding - Completed ✓
2024-03-30 02:50:17,085 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 02:50:35,131 - BERTopic - Dimensionality - Completed ✓
2024-03-30 02:50:35,132 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 02:50:35,831 - BERTopic - Cluster - Completed ✓
2024-03-30 02:50:35,835 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 02:50:45,625 - BERTopic - Representation - Completed ✓
2024-03-30 02:50:48,857 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

...

19


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2024-03-30 03:15:33,675 - BERTopic - Embedding - Completed ✓
2024-03-30 03:15:33,676 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 03:15:48,505 - BERTopic - Dimensionality - Completed ✓
2024-03-30 03:15:48,507 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 03:15:49,205 - BERTopic - Cluster - Completed ✓
2024-03-30 03:15:49,209 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 03:15:59,869 - BERTopic - Representation - Completed ✓
2024-03-30 03:16:02,726 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2024-03-30 03:16:12,622 - BERTopic - Embedding - Completed ✓
2024-03-30 03:16:12,623 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 03:16:27,485 - BERTopic - Dimensionality - Completed ✓
2024-03-30 03:16:27,487 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 03:16:29,924 - BERTopic - Cluster - Completed ✓
2024-03-30 03:16:29,928 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 03:16:39,144 - BERTopic - Representation - Completed ✓
